[← 04 - JOINs](<04-joins.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 05 - Subqueries to CTEs: name the step before you nest the next one

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/05-subqueries-to-ctes.ipynb)

`04 - JOINs` ended with a credit-limit report that counted Ada's limit twice, once for every order she'd placed, and the fix was to collapse orders down to one row per customer before adding anything up. Now the business wants two more columns on that report: whether each customer has already spent more than half their credit limit, and how each customer's spending compares to the average among customers who've placed at least one order. Both depend on the same per-customer total, calculated once and compared against twice. Write that calculation in one place and you can build almost anything on top of it; write it out twice, buried inside two levels of parentheses, and you get a query nobody, including you in three weeks, wants to open again.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** use a scalar subquery as a single value and a derived table as a temporary table, rewrite a nested query as a chain of named CTEs, tell a correlated subquery from an uncorrelated one and spot the silent bug when one should have been the other, and walk a hierarchy with a recursive CTE.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 5 of the All About Data & More curriculum](https://aaboutdata.substack.com/p/sql-ctes-name-the-step-before-you), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 10 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. Same customers, more orders

These are the four customers from `04 - JOINs`, with two more orders since then: Chidi has placed one, and Dami still has none. That absence matters here too, since a customer with no orders shouldn't quietly disappear from a spending report just because there's nothing to sum. Run the setup once; for a clean start later, run `%sql_reset` in a new cell and then run this cell again.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 2. A subquery is a query inside a query

A **subquery** is a `SELECT` nested inside another SQL statement. The simplest kind, a **scalar subquery**, returns exactly one row and one column, so it can stand in anywhere a single value would go. Which orders came in above the average order amount?

In [3]:
%%sql
SELECT
    order_id,
    customer_id,
    amount
FROM orders
WHERE amount > (SELECT AVG(amount) FROM orders)
ORDER BY
    order_id;

order_id,customer_id,amount
104,2,90
105,3,140


The average order amount is 82, and the inner `SELECT` is worked out first, so the outer query behaves as if you'd typed `WHERE amount > 82`. The [SQLite expression reference](https://www.sqlite.org/lang_expr.html) is explicit about what makes a subquery usable this way: a `SELECT` that returns one column is a scalar subquery, and one that returns more than one column is a different thing, a row value, restricted to comparisons and `UPDATE` clauses.

## 3. Build the report with one nested subquery

The credit-limit question needs a per-customer total before it needs anything else, so put that calculation in a subquery and treat it as a table:

In [4]:
%%sql
SELECT
    customer_id,
    customer_name,
    total_spent,
    credit_limit,
    total_spent > credit_limit / 2.0 AS over_half_limit
FROM (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
) AS customer_totals
ORDER BY
    customer_id;

customer_id,customer_name,total_spent,credit_limit,over_half_limit
1,Ada,100,100,1
2,Bola,170,200,1
3,Chidi,140,300,0
4,Dami,0,200,0


`customer_totals` is called a **derived table**: a subquery given an alias and queried in `FROM` exactly like a real table, though it exists only for the length of this one statement. `COALESCE` turns Dami's missing sum into a real zero, which is what lets her still show up with a defined `over_half_limit` instead of a blank. In SQLite a comparison like `total_spent > credit_limit / 2.0` gives `1` for true and `0` for false, so Ada and Bola have already crossed half their limit, and Chidi and Dami haven't.

## 4. Stack a second question and the nesting turns ugly

Add the second column, how each customer compares to the average total among customers who've actually ordered something, and the same `customer_totals` logic has to appear a second time, because the average needs its own aggregation over the same base data:

In [5]:
%%sql
SELECT
    customer_id,
    customer_name,
    total_spent,
    credit_limit,
    total_spent > credit_limit / 2.0 AS over_half_limit,
    total_spent > (
        SELECT AVG(total_spent)
        FROM (
            SELECT
                c2.customer_id,
                COALESCE(SUM(o2.amount), 0) AS total_spent
            FROM customers AS c2
            INNER JOIN orders AS o2
                ON c2.customer_id = o2.customer_id
            GROUP BY
                c2.customer_id
        )
    ) AS above_average_spender
FROM (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
) AS customer_totals
ORDER BY
    customer_id;

customer_id,customer_name,total_spent,credit_limit,over_half_limit,above_average_spender
1,Ada,100,100,1,0
2,Bola,170,200,1,1
3,Chidi,140,300,0,1
4,Dami,0,200,0,0


The average total among the three ordering customers is 136.67, so Bola and Chidi clear it while Ada, despite already being over half her own limit, hasn't. That's a real and correct result, but look at what it cost: the same join-and-group logic is now written out twice, once for the outer report and again, with different aliases, inside the average. Change how "total spent" is defined, say to exclude a future refunds table, and there are two places to remember to change it, not one.

(Notice too that the inner derived table, the one inside the average, has no alias. SQLite lets that go; section 10 shows SQL Server refusing it.)

## 5. Name it once with a CTE

A **common table expression**, or CTE, is a named subquery defined with `WITH` before the main query. Any part of the same statement that follows can reference it by name, instead of the query text getting copy-pasted:

In [6]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
),
ordering_customers AS (
    SELECT
        customer_id,
        total_spent
    FROM customer_totals
    WHERE total_spent > 0
)
SELECT
    customer_id,
    customer_name,
    total_spent,
    credit_limit,
    total_spent > credit_limit / 2.0 AS over_half_limit,
    total_spent > (SELECT AVG(total_spent) FROM ordering_customers) AS above_average_spender
FROM customer_totals
ORDER BY
    customer_id;

customer_id,customer_name,total_spent,credit_limit,over_half_limit,above_average_spender
1,Ada,100,100,1,0
2,Bola,170,200,1,1
3,Chidi,140,300,0,1
4,Dami,0,200,0,0


Same result as the nested version, but `customer_totals` is written exactly once, and both `ordering_customers` and the final `SELECT` reference it by name. `ordering_customers` is itself a CTE built on top of `customer_totals`, which is what **stacking** CTEs means: each one is a named, readable step, defined in the order you think about the problem rather than the order a nested subquery forces you to unwind it in.

<p align="center">
  <img src="graphics/05_nested_vs_cte.png" width="300" alt="Nested: the customer_totals calculation is typed twice and the final SELECT reads both copies; CTE: WITH customer_totals is defined once, ordering_customers builds on it, and the final SELECT references both">
</p>

## 6. How SQLite actually treats a CTE

`WITH` doesn't promise that a CTE runs once no matter how many times you reference it. By default, [SQLite is free to either substitute the CTE's text at each reference or compute it once and reuse the result](https://www.sqlite.org/lang_with.html), whichever its query planner thinks will work best. If you need to force one behavior, SQLite 3.35.0, released 2021-03-12, added `AS MATERIALIZED (...)` and `AS NOT MATERIALIZED (...)`:

In [7]:
%%sql
WITH customer_totals AS MATERIALIZED (
    SELECT
        c.customer_id,
        c.customer_name,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)
SELECT
    customer_name,
    total_spent
FROM customer_totals
WHERE total_spent > (SELECT AVG(total_spent) FROM customer_totals)
ORDER BY
    customer_id;

customer_name,total_spent
Bola,170
Chidi,140


The keyword changed how SQLite runs the CTE, not what it returns: the average across all four customers here is 102.5, and Bola and Chidi clear it either way. An install older than 3.35.0 doesn't have either keyword at all, and neither does SQL Server, so a query written assuming you can force materialization won't run there. It only matters if you're relying on a CTE running once for performance, in which case check your engine's version and its query plan rather than assuming.

<p align="center">
  <img src="graphics/05_cte_reuse.png" width="560" alt="customer_totals is defined once and referenced twice; SQLite may inline or materialize it, and AS MATERIALIZED or AS NOT MATERIALIZED, from SQLite 3.35.0, forces one behavior">
</p>

## 7. Common mistake: writing WITH twice

A beginner adding a second CTE sometimes reaches for a second `WITH` keyword instead of a comma. This cell is meant to fail:

In [8]:
%%sql --expect-error
WITH first_cte AS (SELECT 1 AS x)
WITH second_cte AS (SELECT 2 AS y)
SELECT
    x,
    y
FROM first_cte, second_cte;

Error (this cell is meant to fail): near "WITH": syntax error


One `WITH` keyword introduces the whole list; every CTE after the first is separated by a comma:

In [9]:
%%sql
WITH first_cte AS (SELECT 1 AS x),
     second_cte AS (SELECT 2 AS y)
SELECT
    x,
    y
FROM first_cte, second_cte;

x,y
1,2


## 8. Common mistake: a subquery that forgets to correlate

Suppose you'd written the half-limit comparison as a scalar subquery instead of a plain column expression, and forgot to tie it back to the row you're comparing:

In [10]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
)
SELECT
    customer_id,
    customer_name,
    total_spent
FROM customer_totals
WHERE total_spent > (SELECT credit_limit / 2.0 FROM customers)
ORDER BY
    customer_id;

customer_id,customer_name,total_spent
1,Ada,100
2,Bola,170
3,Chidi,140


This doesn't error, and that's what makes it dangerous. The inner `SELECT credit_limit / 2.0 FROM customers` returns four rows, one per customer, and the [SQLite documentation is direct about what happens next](https://www.sqlite.org/lang_expr.html): "the value of a subquery expression is the first row of the result from the enclosed SELECT statement." Every other row is silently ignored. Here that first row happens to be Ada's own half-limit, 50, so the filter quietly became `total_spent > 50` for everyone, which is why Chidi shows up despite being nowhere near his own 150. Which row counts as "first" isn't something the query defines either, so relying on it, even by accident, is relying on nothing.

The fix is to **correlate** the subquery, tying it back to the outer row by customer ID, so it's worked out once per customer instead of once for the whole table:

In [11]:
%%sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.credit_limit,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.credit_limit
)
SELECT
    ct.customer_id,
    ct.customer_name,
    ct.total_spent
FROM customer_totals AS ct
WHERE ct.total_spent > (
    SELECT c2.credit_limit / 2.0
    FROM customers AS c2
    WHERE c2.customer_id = ct.customer_id
)
ORDER BY
    ct.customer_id;

customer_id,customer_name,total_spent
1,Ada,100
2,Bola,170


That matches `over_half_limit` from earlier: only Ada and Bola. A subquery that references a column from the outer query, like `ct.customer_id` here, is a **correlated subquery**, evaluated fresh for each outer row.

<p align="center">
  <img src="graphics/05_subquery_vocabulary.png" width="720" alt="Scalar subquery: one row and column standing in for a value; derived table: an aliased subquery in FROM; CTE: a named subquery via WITH; correlated subquery: references the outer row and runs once per row">
</p>

## 9. A CTE can reference itself

A **recursive CTE** can refer to its own name inside its own definition, which is how you walk a hierarchy without knowing its depth in advance. Here's a small org chart, separate from the customer data:

In [12]:
%%sql
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    manager_id INTEGER
);
INSERT INTO employees VALUES
    (1, 'Ngozi', NULL),
    (2, 'Femi', 1),
    (3, 'Tunde', 1),
    (4, 'Zainab', 2);

4 rows inserted


In [13]:
%%sql
WITH RECURSIVE org_tree AS (
    SELECT
        employee_id,
        employee_name,
        manager_id,
        1 AS depth
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    SELECT
        e.employee_id,
        e.employee_name,
        e.manager_id,
        ot.depth + 1
    FROM employees AS e
    INNER JOIN org_tree AS ot
        ON e.manager_id = ot.employee_id
)
SELECT
    employee_id,
    employee_name,
    depth
FROM org_tree
ORDER BY
    depth,
    employee_name;

employee_id,employee_name,depth
1,Ngozi,1
2,Femi,2
3,Tunde,2
4,Zainab,3


The first part of the `UNION ALL` finds the **anchor**, Ngozi, who has no manager, and the second part repeatedly joins employees onto the rows `org_tree` has already produced, adding one level each pass, until nobody new matches. It's a genuinely different tool from the report above, worth knowing for hierarchies, dependency graphs, and folder trees.

One detail matters the moment you move between engines. The SQL standard requires the `RECURSIVE` keyword on a CTE that references itself, and PostgreSQL and MySQL enforce it, while SQLite runs the query with or without it. SQL Server is the odd one out in the other direction: it doesn't accept the keyword at all, and section 10 shows the error. So the rule is to write `RECURSIVE` for SQLite, PostgreSQL, and MySQL, and leave it out for SQL Server.

## 10. Scale it up: the same ideas in AdventureWorks

Five orders make every subquery checkable by hand, and real data is where a forgotten correlation or a copied calculation does damage nobody notices. This section runs the lesson's ideas against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server. Three things change from here on: tables live in schemas (`Sales.SalesOrderHeader`), the dialect is T-SQL, and the results are far too big to eyeball, which is the whole point.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [14]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### A scalar subquery at scale

In [15]:
%%tsql
SELECT COUNT(*) AS orders_above_average
FROM Sales.SalesOrderHeader
WHERE TotalDue > (SELECT AVG(TotalDue) FROM Sales.SalesOrderHeader);

orders_above_average
4012


4,012 of 31,465 orders are above the average of about $3,916, which says something about the shape of the data before you've looked at a single row: a small number of very large orders pulls the average up, so most orders sit below it.

### The silent first-row bug, refused

Section 8's uncorrelated subquery returned four rows and SQLite quietly used the first. Here's the same mistake on SQL Server, comparing every order to "the" order total of customer 29825, who placed twelve orders. This cell is meant to fail:

In [16]:
%%tsql --expect-error
SELECT COUNT(*) AS bigger_orders
FROM Sales.SalesOrderHeader
WHERE TotalDue > (
    SELECT TotalDue
    FROM Sales.SalesOrderHeader
    WHERE CustomerID = 29825
);

Error (this cell is meant to fail): Subquery returned more than 1 value. This is not permitted when the subquery follows =, !=, <, <= , >, >= or when the subquery is used as an expression.


SQL Server refuses to guess: a subquery used as a single value must return at most one row, and twelve rows is an error, not a first-row pick. That's the safer behavior, and a good reason to test a query on more than one engine when you can.

### A derived table needs a name

Section 4 left the inner derived table unaliased and SQLite didn't mind. SQL Server does. This cell is meant to fail:

In [17]:
%%tsql --expect-error
SELECT AVG(customer_total) AS average_customer_total
FROM (
    SELECT SUM(TotalDue) AS customer_total
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
);

Error (this cell is meant to fail): Incorrect syntax near ';'.


SQL Server stops with a syntax error, which is its unhelpful way of saying the derived table has no alias. Give it one and it runs:

In [18]:
%%tsql
SELECT AVG(customer_total) AS average_customer_total
FROM (
    SELECT SUM(TotalDue) AS customer_total
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
) AS customer_totals;

average_customer_total
6444.7296


### The report as CTEs

Here's the lesson's pattern on real data: one CTE for each customer's total, one for the average of those totals, and a final query that compares the two.

In [19]:
%%tsql
WITH customer_totals AS (
    SELECT
        CustomerID,
        COUNT(*) AS orders,
        SUM(TotalDue) AS total_spent
    FROM Sales.SalesOrderHeader
    GROUP BY
        CustomerID
),
average_total AS (
    SELECT AVG(total_spent) AS average_spent
    FROM customer_totals
)
SELECT TOP 5
    ct.CustomerID,
    ct.orders,
    ct.total_spent,
    ct.total_spent / a.average_spent AS times_the_average
FROM customer_totals AS ct
CROSS JOIN average_total AS a
ORDER BY
    ct.total_spent DESC;

CustomerID,orders,total_spent,times_the_average
29818,12,989184.0820,153.4872
29715,12,961675.8596,149.2189
29722,12,954021.9235,148.0313
30117,12,919801.8188,142.7215
29614,12,901346.8560,139.8579


The biggest customer has spent more than 150 times what the average customer has. `CROSS JOIN` attaches the single row of `average_total` to every customer, the one place a deliberate cross join is useful (`04 - JOINs` showed what an accidental one does).

### A recursive CTE on a real hierarchy

AdventureWorks records which parts each product is assembled from in `Production.BillOfMaterials`: a row says "assembly X uses component Y". A bike is made of components, which are themselves made of components, to an unknown depth, which is exactly what a recursive CTE is for. Note there's no `RECURSIVE` keyword, since SQL Server doesn't accept it:

In [20]:
%%tsql
WITH parts AS (
    SELECT
        b.ComponentID,
        1 AS depth
    FROM Production.BillOfMaterials AS b
    WHERE b.ProductAssemblyID = 749
      AND b.EndDate IS NULL

    UNION ALL

    SELECT
        b.ComponentID,
        p.depth + 1
    FROM Production.BillOfMaterials AS b
    INNER JOIN parts AS p
        ON b.ProductAssemblyID = p.ComponentID
    WHERE b.EndDate IS NULL
)
SELECT
    depth,
    COUNT(*) AS components
FROM parts
GROUP BY
    depth
ORDER BY
    depth;

depth,components
1,14
2,50
3,21
4,4


Product 749, the Road-150 Red, 62, is built from 14 components directly, and walking down the tree finds 50 more one level below, 21 below those, and 4 at the bottom. (`EndDate IS NULL` keeps only the parts list that's currently in use.) And here's what happens with the keyword the standard asks for. This cell is meant to fail:

In [21]:
%%tsql --expect-error
WITH RECURSIVE numbers AS (
    SELECT 1 AS n
)
SELECT n
FROM numbers;

Error (this cell is meant to fail): Incorrect syntax near 'numbers'.


## 11. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/05-subqueries-to-ctes-solutions.ipynb>), predicting the result first.

1. **Find the top spender.** Using `customer_totals` from section 5, return the single customer with the highest total spent. Predict the name before you run it.
2. **Customers with no orders, without a join.** Return the customers who have never ordered, using a subquery in `WHERE` rather than a `LEFT JOIN`.
3. **Count each manager's direct reports.** For every employee in the org chart, return their name and how many people report directly to them, using a correlated subquery. Everyone should appear, including people with nobody reporting to them.
4. **Stretch, on AdventureWorks.** Using CTEs, find how many customers have spent more than the average customer, out of all customers who've placed an order.

## Before you trust a subquery

- Name a step with a CTE as soon as you need it twice, or as soon as you can't read the query top to bottom.
- A scalar subquery must return one row. If it can return more, decide which one you mean and say so, usually by correlating it.
- Give every derived table an alias; some engines require it.
- Check whether the outer query should reference the inner one. A subquery that ought to be correlated and isn't runs without an error.
- Write `RECURSIVE` for SQLite, PostgreSQL, and MySQL, and leave it out for SQL Server.

## Go deeper

- **[The WITH clause](https://www.sqlite.org/lang_with.html)**, SQLite. Ordinary and recursive CTEs, the `MATERIALIZED` hints, and the history behind SQLite's relaxed `RECURSIVE` rule.
- **[Literate SQL: WITH](https://modern-sql.com/feature/with)**, Markus Winand. Why CTEs exist and how each major engine supports them, written by the author of *SQL Performance Explained*.
- **[WITH common_table_expression](https://learn.microsoft.com/en-us/sql/t-sql/queries/with-common-table-expression-transact-sql)** and **[Subqueries](https://learn.microsoft.com/en-us/sql/relational-databases/performance/subqueries)**, Microsoft Learn. The SQL Server references behind section 10, with more examples of recursive CTEs.
- **[PostgreSQL: WITH Queries](https://www.postgresql.org/docs/current/queries-with.html)**. A thorough treatment of recursive queries, including how to stop one that would loop forever.
- **Practice:** [SQLZoo: SELECT within SELECT](https://sqlzoo.net/wiki/SELECT_within_SELECT_Tutorial) for subqueries, and [DataLemur's CTE vs subquery tutorial](https://datalemur.com/sql-tutorial/sql-cte-subquery) for the interview framing.

For interview questions on this topic with model answers, see the [study guide](<study-guides/05-subqueries-to-ctes.md>).

## What's next

A CTE doesn't let you calculate anything a subquery couldn't; what it buys you is a name, one place to define a step and as many places as you need to reference it. `06 - Window Functions, Part 1` closes a gap this notebook couldn't close cleanly: comparing each row to a group total while still keeping every row, something we've only managed here by joining a summary back onto itself.

---

[← 04 - JOINs](<04-joins.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [06 - Window Functions, Part 1 →](<06-window-functions-part-1.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).